# 1、流式调用(stream)

举例：

In [1]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

# 加载配置文件
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

# 获取大模型
model = init_chat_model(
    model="openai:gpt-5.4-mini",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL,
)

In [2]:
for chunk in model.stream("帮我解释一下什么是人工智能？"):
    print(chunk.text,end="",flush=True)

人工智能（Artificial Intelligence，简称 AI）是指：**让机器像人一样“思考”和“行动”**的技术。

更具体地说，人工智能让电脑或机器具备一些原本需要人类智能才能完成的能力，比如：

- **识别图像**：比如手机人脸解锁、图片分类  
- **理解语言**：比如语音助手、聊天机器人  
- **学习规律**：比如根据数据预测天气、推荐视频  
- **做决策**：比如自动驾驶、智能客服  
- **生成内容**：比如写文章、作图、翻译

### 一个简单例子
如果你给电脑很多猫和狗的图片，让它自己找出区别，最后它能判断一张新图片是猫还是狗，这就是人工智能的一种应用。

### 人工智能不是“真的像人一样有意识”
虽然它能完成一些很聪明的任务，但它通常：
- 没有真正的情感
- 没有自我意识
- 不会像人一样理解世界

它更多是通过**算法 + 数据 + 计算能力**来模仿人类的某些智能行为。

### 常见的人工智能类型
- **弱人工智能**：专注于某一项任务，比如语音识别、翻译
- **强人工智能**：像人一样拥有全面智能和自我意识，目前还没有真正实现

如果你愿意，我还可以继续给你讲：
1. **人工智能的发展历史**
2. **机器学习和人工智能有什么区别**
3. **AI 在生活中的具体应用**

# 2、批量调用(batch)

举例1：一次性接收所有的响应

In [3]:
messages = [
    "你好，你是谁？",
    "2 + 3 * 5 = ?",
    "中国首都在哪里？"
]

responses = model.batch(messages)

for response in responses:
    print(response)

content='你好！我是 ChatGPT，一个由 OpenAI 训练的 AI 助手。  \n我可以帮你解答问题、写作、翻译、总结、编程、头脑风暴等。\n\n如果你愿意，可以直接告诉我你现在想做什么。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 10, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 5, 'engine_ttft_ms': 39, 'engine_ttlt_ms': 324, 'pre_inference_ms': 82, 'service_tbt_ms': 5, 'service_ttft_ms': 274, 'service_ttlt_ms': 549, 'total_duration_ms': 475, 'user_visible_ttft_ms': 191}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DhGEjuBJkZ7YUrlxNhMfosn43UvTJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e40c1-8b1c-73b3-bd0f-52397783b655-0' tool_calls=[] invalid_tool_calls=[] usage_metadata

举例2：按照完成的顺序接收响应

In [5]:
messages = [
    "你好，你是谁？",
    "2 + 3 * 5 = ?",
    "中国首都在哪里？"
]

responses = model.batch_as_completed(messages)

for response in responses:
    print(response)

(2, AIMessage(content='中国的首都是**北京**。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 11, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 4, 'engine_ttft_ms': 35, 'engine_ttlt_ms': 87, 'pre_inference_ms': 115, 'service_tbt_ms': 2, 'service_ttft_ms': 377, 'service_ttlt_ms': 405, 'total_duration_ms': 280, 'user_visible_ttft_ms': 262}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DhGG0AeKhI9MZOOJPo00OprO4k1yL', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e40c2-c2d8-73a1-b330-68ce120ab7dd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 12, 'total_tokens': 

举例3：性能对比

使用batch

In [6]:
# 准备多个输入
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]

# ✅ 批量调用（高效）
import time

start = time.time()
responses = model.batch(inputs)
batch_time = time.time() - start

print("批量调用结果：")
for i, response in enumerate(responses):
    print(f"{i+1}. {response.content}")
print(f"耗时: {batch_time:.2f}秒\n")


批量调用结果：
1. “Spring has come.”
2. “夏天很热” can be translated into English as:

**“Summer is very hot.”**
3. “秋天落叶”可以翻译成英文 **“autumn leaves”**，也可以说 **“fall leaves”**（美式英语里更常用）。

如果你想表达“秋天里的落叶”这个意思，**“fallen leaves in autumn”** 也很自然。
4. “Winter is snowy.”  
If you mean the more natural phrase, it is: **“It snows in winter.”**
耗时: 2.24秒



作为对比，演示循环调用invoke()

In [7]:
# ❌ 循环调用（低效，仅用于对比）
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]

start = time.time()
loop_responses = []
for inp in inputs:
    response = model.invoke(inp)
    loop_responses.append(response)
loop_time = time.time() - start

for i, response in enumerate(responses):
    print(f"{i+1}. {response.content}")

print(f"循环调用耗时: {loop_time:.2f}秒")

print(f"批量调用节省: {((loop_time - batch_time) / loop_time * 100):.1f}%")

1. “Spring has come.”
2. “夏天很热” can be translated into English as:

**“Summer is very hot.”**
3. “秋天落叶”可以翻译成英文 **“autumn leaves”**，也可以说 **“fall leaves”**（美式英语里更常用）。

如果你想表达“秋天里的落叶”这个意思，**“fallen leaves in autumn”** 也很自然。
4. “Winter is snowy.”  
If you mean the more natural phrase, it is: **“It snows in winter.”**
循环调用耗时: 5.10秒
批量调用节省: 56.2%
